# 03 - DV01, KRD, and Scenario Comparison
Build a portfolio risk report and compare total DV01 under base/up/down curve shifts.

### Conceptual roadmap
1. Calculate instrument-level DV01 and key-rate buckets.
2. Aggregate to portfolio totals.
3. Re-run totals under parallel curve shifts for scenario comparison.

In [ ]:
from __future__ import annotations
from datetime import date
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from fixed_income_toolkit.io import read_portfolio_csv, read_treasuries_csv
from fixed_income_toolkit.curve import bootstrap_spot_curve
from fixed_income_toolkit.risk import portfolio_risk_report
from fixed_income_toolkit.types import CurvePoint

In [ ]:
settlement = date(2026, 9, 15)
bonds = read_portfolio_csv(repo_root / "data" / "sample" / "portfolio.csv", settlement=settlement)
treasuries = read_treasuries_csv(repo_root / "data" / "sample" / "treasuries.csv")
base_curve = bootstrap_spot_curve(treasuries)

rows, summary = portfolio_risk_report(bonds, base_curve)
risk_df = pd.DataFrame([{
    "instrument_id": r.instrument_id,
    "dv01": r.dv01,
    "krd_2y": r.krd_2y,
    "krd_5y": r.krd_5y,
    "krd_10y": r.krd_10y,
    "krd_30y": r.krd_30y,
} for r in rows])

risk_df

### Step 2: Scenario engine
A simple parallel shift helper is used to generate shocked curves and compare total DV01/KRD outcomes across scenarios.

In [ ]:
def shift_curve(points: list[CurvePoint], shift_bps: float) -> list[CurvePoint]:
    shifted = []
    shift = shift_bps / 10000.0
    for p in points:
        new_zero = max(0.0, p.zero_rate + shift)
        freq = 2
        new_df = (1.0 + new_zero / freq) ** (-freq * p.tenor_years)
        shifted.append(CurvePoint(tenor_years=p.tenor_years, zero_rate=new_zero, discount_factor=new_df))
    return shifted

scenario_defs = {"base": 0.0, "up_25bp": 25.0, "down_25bp": -25.0}
scenario_rows = []
for name, shift in scenario_defs.items():
    curve = shift_curve(base_curve, shift)
    _, s = portfolio_risk_report(bonds, curve)
    scenario_rows.append({
        "scenario": name,
        "curve_shift_bps": shift,
        "total_dv01": s.total_dv01,
        "krd_2y": s.total_krd_2y,
        "krd_5y": s.total_krd_5y,
        "krd_10y": s.total_krd_10y,
        "krd_30y": s.total_krd_30y,
    })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(risk_df["instrument_id"], risk_df["dv01"], color="tab:blue")
ax[0].set_title("Instrument DV01")
ax[0].set_ylabel("DV01")
ax[0].tick_params(axis="x", rotation=25)

ax[1].plot(scenario_df["scenario"], scenario_df["total_dv01"], marker="o", color="tab:red")
ax[1].set_title("Portfolio DV01 by Scenario")
ax[1].set_ylabel("Total DV01")
ax[1].grid(alpha=0.3)

plt.tight_layout()